# Failure audits and learnability diagnostics

Historical research source, original zero-based cells **31?37**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 31

Position 32 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import numpy as np
import torch

from pathlib import Path
from torchcodec.decoders import AudioDecoder
from jiwer import cer, wer
from tqdm.auto import tqdm

# ==========================================
# 1. VERIFY CHECKPOINT
# ==========================================

EVAL_STEP = 500
BLANK_ID = tokenizer.pad_token_id

assert successful_updates == EVAL_STEP
assert (
    CHECKPOINT_DIR / "step_000500" / "COMPLETE"
).exists()

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]

eval_records = read_jsonl(BASELINE_EVAL_PATH)

assert len(eval_records) == 12

# ==========================================
# 2. LOAD PREVIOUS EVALUATIONS
# ==========================================

previous_metrics = {}

for step, filename in [
    (0, "baseline_step0_metrics.json"),
    (50, "eval_step_000050_metrics.json"),
    (200, "eval_step_000200_metrics.json")
]:
    with (DATA_DIR / filename).open(encoding="utf-8") as f:
        previous_metrics[step] = json.load(f)

previous_predictions = {}

for step, filename in [
    (0, "baseline_step0_predictions_12.jsonl"),
    (50, "eval_step_000050_predictions_12.jsonl"),
    (200, "eval_step_000200_predictions_12.jsonl")
]:
    previous_predictions[step] = {
        r["id"]: r
        for r in read_jsonl(DATA_DIR / filename)
    }

expected_ids = {r["id"] for r in eval_records}

for step in previous_predictions:
    assert set(previous_predictions[step]) == expected_ids

# ==========================================
# 3. INFERENCE HELPER
# ==========================================

def inspect_asr_example(example):

    batch = ctc_collator([example])
    batch = {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)

    logits = outputs.logits[0].float()
    loss = float(outputs.loss.item())

    assert torch.isfinite(logits).all()
    assert np.isfinite(loss)

    probabilities = logits.softmax(dim=-1)
    token_ids = probabilities.argmax(dim=-1)

    predicted = processor.batch_decode(
        token_ids.unsqueeze(0).cpu()
    )[0]

    predicted = normalize_amharic_ctc(predicted)

    blank_fraction = (
        (token_ids == BLANK_ID)
        .float()
        .mean()
        .item()
    )

    blank_probability = (
        probabilities[:, BLANK_ID]
        .mean()
        .item()
    )

    nonblank = probabilities.clone()
    nonblank[:, BLANK_ID] = 0.0

    strongest_nonblank = nonblank.max(dim=-1).values

    mean_nonblank = strongest_nonblank.mean().item()
    peak_nonblank = strongest_nonblank.max().item()

    mean_token_probs = nonblank.mean(dim=0)

    top_token_ids = torch.topk(
        mean_token_probs,
        k=5
    ).indices.tolist()

    top_tokens = [
        tokenizer.convert_ids_to_tokens(token_id)
        for token_id in top_token_ids
    ]

    return {
        "id": example["id"],
        "prediction": predicted,
        "ctc_loss": loss,
        "blank_fraction": blank_fraction,
        "blank_probability": blank_probability,
        "mean_strongest_nonblank_probability": mean_nonblank,
        "peak_nonblank_probability": peak_nonblank,
        "top_nonblank_tokens": top_tokens
    }

# ==========================================
# 4. EVALUATE FIXED VALIDATION RECORDINGS
# ==========================================

model.eval()

validation_results = []
seen_train_results = []

try:

    for record in tqdm(
        eval_records,
        desc="Step-500 validation"
    ):

        path = EVAL_AUDIO_DIR / f"{record['id']}.mp3"
        assert path.exists(), path

        decoder = AudioDecoder(
            str(path),
            sample_rate=SAMPLE_RATE,
            num_channels=1
        )

        samples = decoder.get_all_samples()

        waveform = (
            samples.data[0]
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        features = processor.feature_extractor(
            waveform,
            sampling_rate=SAMPLE_RATE
        )

        reference = record["transcription"]

        example = {
            "id": record["id"],
            "input_values": features.input_values[0],
            "labels": tokenizer.encode(
                reference,
                add_special_tokens=False
            )
        }

        result = inspect_asr_example(example)

        result["reference"] = reference
        result["cer"] = cer(reference, result["prediction"])
        result["wer"] = wer(reference, result["prediction"])

        validation_results.append(result)

    # ======================================
    # 5. CHECK FOUR SEEN TRAINING EXAMPLES
    # ======================================

    for index in training_order[:4]:

        example = train_dataset[index]
        result = inspect_asr_example(example)

        result["reference"] = train_records[index][
            "transcription"
        ]

        result["cer"] = cer(
            result["reference"],
            result["prediction"]
        )

        seen_train_results.append(result)

finally:
    model.train()

# ==========================================
# 6. CALCULATE EVALUATION METRICS
# ==========================================

references = [
    r["reference"] for r in validation_results
]

predictions = [
    r["prediction"] for r in validation_results
]

metrics = {
    "training_step": EVAL_STEP,
    "evaluation_recordings": len(validation_results),
    "cer": cer(references, predictions),
    "wer": wer(references, predictions),
    "mean_ctc_loss": float(np.mean([
        r["ctc_loss"] for r in validation_results
    ])),
    "mean_blank_fraction": float(np.mean([
        r["blank_fraction"] for r in validation_results
    ])),
    "mean_blank_probability": float(np.mean([
        r["blank_probability"] for r in validation_results
    ])),
    "mean_strongest_nonblank_probability": float(
        np.mean([
            r["mean_strongest_nonblank_probability"]
            for r in validation_results
        ])
    ),
    "empty_predictions": sum(
        not r["prediction"] for r in validation_results
    ),
    "mean_prediction_length": float(np.mean([
        len(r["prediction"])
        for r in validation_results
    ]))
}

# ==========================================
# 7. SAVE RESULTS
# ==========================================

predictions_path = (
    DATA_DIR / "eval_step_000500_predictions_12.jsonl"
)

metrics_path = (
    DATA_DIR / "eval_step_000500_metrics.json"
)

train_diagnostics_path = (
    DATA_DIR / "eval_step_000500_seen_train_4.jsonl"
)

for path, records in [
    (predictions_path, validation_results),
    (train_diagnostics_path, seen_train_results)
]:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# ==========================================
# 8. FOUR-CHECKPOINT COMPARISON
# ==========================================

print("\n" + "=" * 72)
print("AMHARIC ASR CHECKPOINT COMPARISON")
print("=" * 72)

print(
    f"{'Step':<10}"
    f"{'CER':>12}"
    f"{'WER':>12}"
    f"{'Blank frames':>17}"
    f"{'Empty':>10}"
)

print("-" * 65)

for step, values in [
    (0, previous_metrics[0]),
    (50, previous_metrics[50]),
    (200, previous_metrics[200]),
    (500, metrics)
]:
    print(
        f"{step:<10}"
        f"{values['cer'] * 100:>11.2f}%"
        f"{values['wer'] * 100:>11.2f}%"
        f"{values['mean_blank_fraction'] * 100:>16.2f}%"
        f"{values['empty_predictions']:>10}"
    )

# ==========================================
# 9. STEP-500 PROBABILITY DIAGNOSTICS
# ==========================================

print("\n" + "=" * 72)
print("STEP-500 VALIDATION DIAGNOSTICS")
print("=" * 72)

print("Mean validation CTC loss:",
      round(metrics["mean_ctc_loss"], 4))

print("Mean blank probability:",
      f"{metrics['mean_blank_probability'] * 100:.2f}%")

print("Mean strongest nonblank probability:",
      f"{metrics['mean_strongest_nonblank_probability'] * 100:.2f}%")

print("Mean prediction length:",
      round(metrics["mean_prediction_length"], 2))

print("Empty predictions:",
      metrics["empty_predictions"], "/12")

# ==========================================
# 10. INSPECT PREVIOUSLY SEEN TRAINING AUDIO
# ==========================================

print("\n" + "=" * 72)
print("SEEN TRAINING RECORDINGS")
print("=" * 72)

for i, result in enumerate(seen_train_results, 1):
    print(f"\nRecording {i}")
    print("CTC loss:", round(result["ctc_loss"], 4))
    print("CER:", f"{result['cer'] * 100:.2f}%")
    print(
        "Blank argmax:",
        f"{result['blank_fraction'] * 100:.2f}%"
    )
    print(
        "Blank probability:",
        f"{result['blank_probability'] * 100:.2f}%"
    )
    print("Top nonblank tokens:", result["top_nonblank_tokens"])
    print("Prediction:", result["prediction"][:120] or "[EMPTY]")

# ==========================================
# 11. SHOW ACTUAL TRANSCRIPTION COMPARISONS
# ==========================================

print("\n" + "=" * 72)
print("BEFORE-AND-AFTER AMHARIC TRANSCRIPTIONS")
print("=" * 72)

for index in [0, 5, 11]:

    current = validation_results[index]
    record_id = current["id"]

    print(f"\n{'-' * 60}")
    print("RECORDING:", index + 1)

    print("REFERENCE:")
    print(current["reference"][:180])

    for step in [0, 50, 200]:
        previous = previous_predictions[step][record_id]

        print(f"\nSTEP {step}:")
        print(previous["prediction"][:180] or "[EMPTY]")

    print("\nSTEP 500:")
    print(current["prediction"][:180] or "[EMPTY]")

    print(
        "Top nonblank tokens:",
        current["top_nonblank_tokens"]
    )

print("\nStep-500 metrics saved:", metrics_path)
print("Step-500 predictions saved:", predictions_path)
print("Seen-training diagnostics:", train_diagnostics_path)
print("\nSTEP-500 EVALUATION COMPLETE")


## Original source cell 32

Position 33 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import numpy as np
import torch

from pathlib import Path
from torchcodec.decoders import AudioDecoder

# ==========================================
# 1. VERIFY MODEL STATE
# ==========================================

assert successful_updates == 500
assert model.training

BLANK_ID = tokenizer.pad_token_id

# Two recordings already used for training.
seen_examples = [
    train_dataset[i]
    for i in training_order[:2]
]

# Two held-out validation recordings.
validation_examples = []

for record in eval_records[:2]:

    path = EVAL_AUDIO_DIR / f"{record['id']}.mp3"

    decoder = AudioDecoder(
        str(path),
        sample_rate=SAMPLE_RATE,
        num_channels=1
    )

    waveform = (
        decoder.get_all_samples()
        .data[0]
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    features = processor.feature_extractor(
        waveform,
        sampling_rate=SAMPLE_RATE
    )

    validation_examples.append({
        "id": record["id"],
        "input_values": features.input_values[0],
        "labels": tokenizer.encode(
            record["transcription"],
            add_special_tokens=False
        )
    })

groups = {
    "SEEN TRAINING": seen_examples,
    "VALIDATION": validation_examples
}

# ==========================================
# 2. DIAGNOSTIC INFERENCE FUNCTION
# ==========================================

def probe_model(example, audio_values):

    modified = {
        "id": example["id"],
        "input_values": np.asarray(
            audio_values,
            dtype=np.float32
        ).copy(),
        "labels": example["labels"]
    }

    batch = ctc_collator([modified])

    batch = {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            output = model(
                **batch,
                output_hidden_states=True
            )

    logits = output.logits[0].float()
    probabilities = logits.softmax(dim=-1)

    assert torch.isfinite(logits).all()
    assert torch.isfinite(output.loss)

    predicted_ids = logits.argmax(dim=-1)

    prediction = processor.batch_decode(
        predicted_ids.unsqueeze(0).cpu()
    )[0]

    # Hidden-state variability across time.
    # Index 0 is the encoder input.
    # Index 24 is the final Transformer output.
    hidden_variation = {}

    for layer_index in [0, 12, 22, 24]:

        hidden = output.hidden_states[
            layer_index
        ][0].float()

        temporal_std = hidden.std(
            dim=0,
            unbiased=False
        ).mean().item()

        rms = hidden.square().mean().sqrt().item()

        hidden_variation[str(layer_index)] = {
            "temporal_std": temporal_std,
            "relative_variation": (
                temporal_std / max(rms, 1e-8)
            )
        }

    result = {
        "ctc_loss": float(output.loss.item()),
        "blank_probability": float(
            probabilities[:, BLANK_ID].mean().item()
        ),
        "blank_argmax": float(
            (predicted_ids == BLANK_ID)
            .float().mean().item()
        ),
        "logit_temporal_std": float(
            logits.std(
                dim=0,
                unbiased=False
            ).mean().item()
        ),
        "prediction": prediction,
        "hidden_variation": hidden_variation
    }

    # Return probabilities for comparison.
    probability_copy = probabilities.cpu()

    del batch, output, logits, probabilities

    return result, probability_copy

# ==========================================
# 3. COMPARE ORIGINAL, REVERSED, SILENCE
# ==========================================

model.eval()

diagnostic_results = []

try:
    for group_name, examples in groups.items():

        print("\n" + "=" * 65)
        print(group_name)
        print("=" * 65)

        for example in examples:

            original = np.asarray(
                example["input_values"],
                dtype=np.float32
            )

            variations = {
                "original": original,
                "reversed": original[::-1].copy(),
                "silence": np.zeros_like(original)
            }

            outputs = {}
            distributions = {}

            for condition, waveform in variations.items():

                result, probs = probe_model(
                    example,
                    waveform
                )

                outputs[condition] = result
                distributions[condition] = probs

            original_loss = outputs["original"]["ctc_loss"]
            reversed_loss = outputs["reversed"]["ctc_loss"]
            silence_loss = outputs["silence"]["ctc_loss"]

            # Total variation distance, averaged over frames.
            # 0 means identical token distributions.
            # 1 means completely disjoint distributions.
            reverse_distance = (
                0.5 * (
                    distributions["original"]
                    - distributions["reversed"]
                ).abs().sum(dim=-1).mean().item()
            )

            silence_distance = (
                0.5 * (
                    distributions["original"]
                    - distributions["silence"]
                ).abs().sum(dim=-1).mean().item()
            )

            diagnostic_results.append({
                "group": group_name,
                "id": example["id"],
                "conditions": outputs,
                "original_vs_reversed_tvd": reverse_distance,
                "original_vs_silence_tvd": silence_distance
            })

            print(f"\nRECORDING: {example['id']}")
            print("-" * 55)

            print(
                f"{'Input':<12}"
                f"{'CTC loss':>12}"
                f"{'Blank prob':>14}"
                f"{'Logit std':>13}"
            )

            for condition, result in outputs.items():
                print(
                    f"{condition:<12}"
                    f"{result['ctc_loss']:>12.4f}"
                    f"{result['blank_probability'] * 100:>13.2f}%"
                    f"{result['logit_temporal_std']:>13.5f}"
                )

            print("\nLoss difference vs original:")
            print(
                "Reversed:",
                round(reversed_loss - original_loss, 4)
            )
            print(
                "Silence:",
                round(silence_loss - original_loss, 4)
            )

            print("\nOutput distribution difference:")
            print(
                "Original vs reversed:",
                round(reverse_distance, 5)
            )
            print(
                "Original vs silence:",
                round(silence_distance, 5)
            )

            print("\nOriginal audio hidden-state variation:")

            for layer, statistics in outputs[
                "original"
            ]["hidden_variation"].items():

                print(
                    f"Layer {layer:>2}: "
                    f"temporal_std="
                    f"{statistics['temporal_std']:.6f} "
                    f"relative="
                    f"{statistics['relative_variation']:.6f}"
                )

            print(
                "\nOriginal prediction:",
                outputs["original"]["prediction"][:100]
                or "[EMPTY]"
            )

finally:
    model.train()

# ==========================================
# 4. SAVE THE DIAGNOSTIC
# ==========================================

DIAGNOSTIC_PATH = (
    DATA_DIR / "acoustic_sensitivity_step500.json"
)

with DIAGNOSTIC_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        diagnostic_results,
        file,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 65)
print("ACOUSTIC SENSITIVITY DIAGNOSTIC COMPLETE")
print("=" * 65)
print("Results saved:", DIAGNOSTIC_PATH)
print("Model weights updated: NO")
print("Optimizer steps added: 0")


## Original source cell 33

Position 34 in the original notebook; original code comments may use different numbering.


In [ ]:

import gc
import json
import torch
import numpy as np

from transformers import Wav2Vec2ForCTC
from torchcodec.decoders import AudioDecoder

# ==========================================
# 1. LOAD ORIGINAL CHECKPOINT SEPARATELY
# ==========================================

assert successful_updates == 500
assert (BASELINE_DIR / "model.safetensors").exists()

original_training_mode = model.training

baseline_probe = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE)

baseline_probe.eval()
model.eval()

LAYERS = [0, 12, 22, 23, 24]

print("REPRESENTATION COMPARISON")
print("-" * 55)
print("Starting model:", BASELINE_DIR)
print("Fine-tuned model: step 500")
print("Layers inspected:", LAYERS)

# ==========================================
# 2. SELECT ONE TRAINING + ONE VALIDATION CLIP
# ==========================================

training_example = train_dataset[training_order[0]]

validation_record = eval_records[0]

decoder = AudioDecoder(
    str(EVAL_AUDIO_DIR / f"{validation_record['id']}.mp3"),
    sample_rate=SAMPLE_RATE,
    num_channels=1
)

waveform = (
    decoder.get_all_samples()
    .data[0]
    .cpu()
    .numpy()
    .astype(np.float32)
)

validation_features = processor.feature_extractor(
    waveform,
    sampling_rate=SAMPLE_RATE
)

validation_example = {
    "id": validation_record["id"],
    "input_values": validation_features.input_values[0],
    "labels": tokenizer.encode(
        validation_record["transcription"],
        add_special_tokens=False
    )
}

examples = {
    "TRAINING": training_example,
    "VALIDATION": validation_example
}

# ==========================================
# 3. PROBE INTERNAL REPRESENTATIONS
# ==========================================

def representation_probe(current_model, example, silence=False):

    batch = ctc_collator([example])

    if silence:
        batch["input_values"] = torch.zeros_like(
            batch["input_values"]
        )

    batch = {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            output = current_model(
                **batch,
                output_hidden_states=True
            )

    states = {}
    statistics = {}

    for layer_index in LAYERS:
        hidden = (
            output.hidden_states[layer_index][0]
            .float()
            .cpu()
            .clone()
        )

        states[layer_index] = hidden

        temporal_std = hidden.std(
            dim=0,
            unbiased=False
        ).mean().item()

        rms = hidden.square().mean().sqrt().item()

        statistics[layer_index] = {
            "temporal_std": temporal_std,
            "relative_variation": temporal_std / max(rms, 1e-8)
        }

    logits = output.logits[0].float()

    probabilities = (
        logits.softmax(dim=-1)
        .cpu()
        .clone()
    )

    result = {
        "ctc_loss": float(output.loss.item()),
        "blank_probability": float(
            probabilities[:, tokenizer.pad_token_id]
            .mean().item()
        ),
        "statistics": statistics
    }

    del batch, output, logits

    return result, states, probabilities


def relative_rms_difference(a, b):
    difference = (a - b).square().mean().sqrt()
    reference = a.square().mean().sqrt().clamp_min(1e-8)
    return float((difference / reference).item())


# ==========================================
# 4. COMPARE BOTH CHECKPOINTS
# ==========================================

all_results = {}

try:
    for group_name, example in examples.items():

        print("\n" + "=" * 65)
        print(group_name)
        print("Recording:", example["id"])
        print("=" * 65)

        baseline, base_states, base_probs = (
            representation_probe(
                baseline_probe, example
            )
        )

        trained, trained_states, trained_probs = (
            representation_probe(
                model, example
            )
        )

        baseline_silence, base_silence_states, base_silence_probs = (
            representation_probe(
                baseline_probe, example, silence=True
            )
        )

        trained_silence, trained_silence_states, trained_silence_probs = (
            representation_probe(
                model, example, silence=True
            )
        )

        print("\nTEMPORAL VARIATION BY LAYER")
        print(
            f"{'Layer':<8}"
            f"{'Step 0':>13}"
            f"{'Step 500':>13}"
            f"{'Ratio':>12}"
        )
        print("-" * 46)

        layer_comparison = {}

        for layer in LAYERS:

            before = baseline["statistics"][layer][
                "relative_variation"
            ]

            after = trained["statistics"][layer][
                "relative_variation"
            ]

            ratio = after / max(before, 1e-8)

            layer_comparison[str(layer)] = {
                "step0_relative_variation": before,
                "step500_relative_variation": after,
                "ratio": ratio
            }

            print(
                f"{layer:<8}"
                f"{before:>13.6f}"
                f"{after:>13.6f}"
                f"{ratio:>12.4f}"
            )

        # Compare the frozen encoder output.
        frozen_difference = relative_rms_difference(
            base_states[22],
            trained_states[22]
        )

        # Compare acoustic sensitivity in final states.
        base_hidden_sensitivity = relative_rms_difference(
            base_states[24],
            base_silence_states[24]
        )

        trained_hidden_sensitivity = relative_rms_difference(
            trained_states[24],
            trained_silence_states[24]
        )

        # Compare output token distributions.
        base_tvd = float(
            0.5 * (
                base_probs - base_silence_probs
            ).abs().sum(dim=-1).mean().item()
        )

        trained_tvd = float(
            0.5 * (
                trained_probs - trained_silence_probs
            ).abs().sum(dim=-1).mean().item()
        )

        print("\nACOUSTIC SENSITIVITY COMPARISON")
        print("-" * 55)

        print(
            "Frozen layer-22 difference (step 0 vs 500):",
            round(frozen_difference, 8)
        )

        print(
            "Final hidden sensitivity, step 0:",
            round(base_hidden_sensitivity, 5)
        )

        print(
            "Final hidden sensitivity, step 500:",
            round(trained_hidden_sensitivity, 5)
        )

        print(
            "Output TVD vs silence, step 0:",
            round(base_tvd, 5)
        )

        print(
            "Output TVD vs silence, step 500:",
            round(trained_tvd, 5)
        )

        print("\nCTC LOSSES")
        print(
            "Step 0: real / silence:",
            round(baseline["ctc_loss"], 4),
            "/",
            round(baseline_silence["ctc_loss"], 4)
        )

        print(
            "Step 500: real / silence:",
            round(trained["ctc_loss"], 4),
            "/",
            round(trained_silence["ctc_loss"], 4)
        )

        all_results[group_name] = {
            "recording_id": example["id"],
            "layers": layer_comparison,
            "frozen_layer22_relative_difference": frozen_difference,
            "step0_final_hidden_sensitivity": base_hidden_sensitivity,
            "step500_final_hidden_sensitivity": trained_hidden_sensitivity,
            "step0_output_tvd_vs_silence": base_tvd,
            "step500_output_tvd_vs_silence": trained_tvd,
            "step0_ctc_loss": baseline["ctc_loss"],
            "step500_ctc_loss": trained["ctc_loss"]
        }

finally:
    # Do not disturb the training model or optimizer.
    model.train(original_training_mode)

    del baseline_probe
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. SAVE DIAGNOSTIC
# ==========================================

OUTPUT_PATH = (
    DATA_DIR / "representation_comparison_step0_vs_500.json"
)

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        all_results,
        file,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 65)
print("REPRESENTATION COMPARISON COMPLETE")
print("=" * 65)
print("Results:", OUTPUT_PATH)
print("Model weights modified: NO")
print("Optimizer steps added: 0")


## Original source cell 34

Position 35 in the original notebook; original code comments may use different numbering.


In [ ]:

import gc
import json
import numpy as np
import torch

from pathlib import Path
from jiwer import cer
from transformers import Wav2Vec2ForCTC

# ==========================================
# 1. VERIFY ORIGINAL MODEL STATE
# ==========================================

assert successful_updates == 500

PROBE_STEPS = 200
PROBE_SEED = 123

PROBE_ENCODER_LR = 2e-5
PROBE_HEAD_LR = 2e-4

# Save RNG state so our regular training can
# continue without altered dropout randomness.
cpu_rng_before = torch.get_rng_state().clone()
cuda_rng_before = torch.cuda.get_rng_state_all()

torch.manual_seed(PROBE_SEED)
torch.cuda.manual_seed_all(PROBE_SEED)

# ==========================================
# 2. SELECT ONE SEEN TRAINING RECORDING
# ==========================================

seen_indices = training_order[:2000]

candidates = [
    i for i in seen_indices
    if 5 <= train_records[i]["duration_seconds"] <= 12
    and 15 <= len(
        tokenizer.encode(
            train_records[i]["transcription"],
            add_special_tokens=False
        )
    ) <= 85
]

if not candidates:
    candidates = [
        i for i in seen_indices
        if len(
            tokenizer.encode(
                train_records[i]["transcription"],
                add_special_tokens=False
            )
        ) >= 15
    ]

assert candidates, "No suitable recording found."

sample_index = min(
    candidates,
    key=lambda i: train_records[i]["duration_seconds"]
)

sample_record = train_records[sample_index]
sample_example = train_dataset[sample_index]

REFERENCE = sample_record["transcription"]

print("SINGLE-EXAMPLE MEMORIZATION TEST")
print("-" * 55)
print("Recording:", sample_record["id"])
print("Duration:", sample_record["duration_seconds"], "seconds")
print("Reference tokens:", len(sample_example["labels"]))
print("Reference:", REFERENCE[:240])

# ==========================================
# 3. LOAD SEPARATE STEP-500 MODEL
# ==========================================

PROBE_CHECKPOINT = (
    CHECKPOINT_DIR / "step_000500"
)

assert (PROBE_CHECKPOINT / "COMPLETE").exists()

probe_model = Wav2Vec2ForCTC.from_pretrained(
    str(PROBE_CHECKPOINT)
).to(DEVICE)

# Match the original trainable architecture.
for parameter in probe_model.parameters():
    parameter.requires_grad = False

for layer in probe_model.wav2vec2.encoder.layers[-2:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True

for parameter in probe_model.lm_head.parameters():
    parameter.requires_grad = True

probe_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

# ==========================================
# 4. CREATE DIAGNOSTIC OPTIMIZER
# ==========================================

probe_encoder_params = [
    p
    for layer in probe_model.wav2vec2.encoder.layers[-2:]
    for p in layer.parameters()
    if p.requires_grad
]

probe_head_params = [
    p
    for p in probe_model.lm_head.parameters()
    if p.requires_grad
]

probe_optimizer = torch.optim.AdamW(
    [
        {
            "params": probe_encoder_params,
            "lr": PROBE_ENCODER_LR
        },
        {
            "params": probe_head_params,
            "lr": PROBE_HEAD_LR
        }
    ],
    weight_decay=0.0
)

probe_scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0
)

# Prepare the recording once, avoiding
# repeated MP3 decoding.
probe_batch = ctc_collator([sample_example])

probe_batch = {
    key: value.to(DEVICE)
    for key, value in probe_batch.items()
}

# ==========================================
# 5. EVALUATION HELPER
# ==========================================

def evaluate_probe():

    probe_model.eval()

    with torch.inference_mode():

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            output = probe_model(**probe_batch)

        logits = output.logits.float()
        assert torch.isfinite(logits).all()

        probabilities = logits.softmax(dim=-1)

        predicted_ids = logits.argmax(dim=-1)

        prediction = processor.batch_decode(
            predicted_ids.cpu()
        )[0]

        prediction = normalize_amharic_ctc(prediction)

        blank_probability = (
            probabilities[..., tokenizer.pad_token_id]
            .mean()
            .item()
        )

        blank_fraction = (
            (predicted_ids == tokenizer.pad_token_id)
            .float()
            .mean()
            .item()
        )

        result = {
            "ctc_loss": float(output.loss.item()),
            "cer": float(cer(REFERENCE, prediction)),
            "blank_probability": float(blank_probability),
            "blank_fraction": float(blank_fraction),
            "prediction": prediction
        }

    return result

# ==========================================
# 6. TRAIN REPEATEDLY ON ONE RECORDING
# ==========================================

probe_history = []
completed_probe_steps = 0

REPORT_STEPS = {
    0, 10, 25, 50, 75, 100, 150, 200
}

try:

    # First measure the unchanged step-500 copy.
    initial = evaluate_probe()

    probe_history.append({
        "step": 0,
        **initial
    })

    print("\nINITIAL PERFORMANCE")
    print("CTC loss:", round(initial["ctc_loss"], 4))
    print("CER:", f"{initial['cer'] * 100:.2f}%")
    print(
        "Blank probability:",
        f"{initial['blank_probability'] * 100:.2f}%"
    )
    print("Prediction:", initial["prediction"][:150] or "[EMPTY]")

    for step in range(1, PROBE_STEPS + 1):

        probe_model.train()

        probe_optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            output = probe_model(**probe_batch)
            loss = output.loss

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite CTC loss at probe step {step}"
            )

        probe_scaler.scale(loss).backward()

        probe_scaler.unscale_(probe_optimizer)

        torch.nn.utils.clip_grad_norm_(
            probe_encoder_params + probe_head_params,
            max_norm=1.0,
            error_if_nonfinite=True
        )

        old_scale = probe_scaler.get_scale()

        probe_scaler.step(probe_optimizer)
        probe_scaler.update()

        if probe_scaler.get_scale() < old_scale:
            raise RuntimeError(
                f"AMP overflow at probe step {step}"
            )

        completed_probe_steps = step

        del output, loss

        if step in REPORT_STEPS:

            result = evaluate_probe()

            probe_history.append({
                "step": step,
                **result
            })

            print(f"\nSTEP {step}/{PROBE_STEPS}")
            print("CTC loss:", round(result["ctc_loss"], 4))
            print("CER:", f"{result['cer'] * 100:.2f}%")
            print(
                "Blank probability:",
                f"{result['blank_probability'] * 100:.2f}%"
            )
            print(
                "Blank argmax:",
                f"{result['blank_fraction'] * 100:.2f}%"
            )
            print(
                "Prediction:",
                result["prediction"][:150] or "[EMPTY]"
            )

            # Early stopping if the model memorizes
            # the example with sufficiently low CER.
            if result["cer"] <= 0.05:
                print("\nMEMORIZATION TARGET REACHED")
                break

    # ======================================
    # 7. SAVE DIAGNOSTIC RESULTS ONLY
    # ======================================

    PROBE_OUTPUT = (
        DATA_DIR / "single_example_overfit_step500.json"
    )

    report = {
        "source_checkpoint": str(PROBE_CHECKPOINT),
        "recording_id": sample_record["id"],
        "duration_seconds": sample_record["duration_seconds"],
        "reference": REFERENCE,
        "probe_encoder_lr": PROBE_ENCODER_LR,
        "probe_head_lr": PROBE_HEAD_LR,
        "completed_steps": completed_probe_steps,
        "history": probe_history
    }

    with PROBE_OUTPUT.open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(
            report,
            file,
            ensure_ascii=False,
            indent=2
        )

    print("\n" + "=" * 65)
    print("SINGLE-EXAMPLE MEMORIZATION RESULTS")
    print("=" * 65)

    print("Starting CER:", f"{initial['cer'] * 100:.2f}%")
    print(
        "Final CER:",
        f"{probe_history[-1]['cer'] * 100:.2f}%"
    )
    print(
        "Starting CTC loss:",
        round(initial["ctc_loss"], 4)
    )
    print(
        "Final CTC loss:",
        round(probe_history[-1]["ctc_loss"], 4)
    )

    print("\nREFERENCE:")
    print(REFERENCE[:300])

    print("\nFINAL PREDICTION:")
    print(probe_history[-1]["prediction"][:300] or "[EMPTY]")

    print("\nResults saved:", PROBE_OUTPUT)

finally:

    # Free only the diagnostic model.
    del probe_model, probe_optimizer, probe_scaler
    del probe_batch

    gc.collect()
    torch.cuda.empty_cache()

    # Restore randomness used by original training.
    torch.set_rng_state(cpu_rng_before)
    torch.cuda.set_rng_state_all(cuda_rng_before)

print("\nMAIN TRAINING MODEL UNCHANGED")
print("Main optimizer remains at step:", successful_updates)


## Original source cell 35

Position 36 in the original notebook; original code comments may use different numbering.


In [ ]:

import copy
import json
import gc

import torch
import torch.nn.functional as F
from jiwer import cer
from transformers import Wav2Vec2ForCTC

# ==========================================
# 1. CONFIGURATION
# ==========================================


# ==========================================
# RESTORE HEAD-ONLY EXPERIMENT DEPENDENCIES
# No previous training state is required.
# ==========================================

from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Check that the necessary components exist.
required_variables = [
    "BASELINE_DIR",
    "DATA_DIR",
    "train_records",
    "train_dataset",
    "processor",
    "tokenizer",
    "ctc_collator",
    "normalize_amharic_ctc",
]

missing = [
    name for name in required_variables
    if name not in globals()
]

assert not missing, (
    "Missing notebook dependencies: "
    + ", ".join(missing)
)

assert Path(BASELINE_DIR).exists(), (
    f"Original baseline not found: {BASELINE_DIR}"
)

# Restore the original diagnostic recording.
TARGET_ID = "6Q0fEziAqyQVIz4J8aws"

sample_index = next(
    i for i, record in enumerate(train_records)
    if record["id"] == TARGET_ID
)

sample_record = train_records[sample_index]
sample_example = train_dataset[sample_index]

print("HEAD-ONLY EXPERIMENT READY")
print("Device:", DEVICE)
print("Original baseline:", BASELINE_DIR)
print("Sample:", sample_record["id"])
print("Duration:", sample_record["duration_seconds"], "seconds")
print("Target tokens:", len(sample_example["labels"]))
print("No step-500 model required.")


HEAD_ONLY_STEPS = 1000
HEAD_ONLY_LR = 1e-3
BLANK_ID = tokenizer.pad_token_id

REPORT_STEPS = {
    0, 25, 50, 100, 200, 350,
    500, 750, 1000
}

assert sample_record["id"] == (
    "6Q0fEziAqyQVIz4J8aws"
)

REFERENCE = sample_record["transcription"]

print("FROZEN-ENCODER MEMORIZATION EXPERIMENT")
print("-" * 55)
print("Recording:", sample_record["id"])
print("Reference:", REFERENCE)
print("Model starting point: STEP ZERO")
print("Trainable component: CTC HEAD ONLY")
print("Maximum updates:", HEAD_ONLY_STEPS)

# ==========================================
# 2. LOAD THE ORIGINAL XLS-R CHECKPOINT
# ==========================================

baseline_model = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE)

baseline_model.eval()

batch = ctc_collator([sample_example])

batch = {
    key: value.to(DEVICE)
    for key, value in batch.items()
}

# ==========================================
# 3. EXTRACT AND CACHE ORIGINAL FEATURES
# ==========================================


# ==========================================
# 3. EXTRACT AND CACHE ORIGINAL FEATURES
# FIXED: Keep trainable parameters outside inference mode
# ==========================================

with torch.inference_mode():

    original_forward = baseline_model(**batch)

    frozen_features = baseline_model.wav2vec2(
        input_values=batch["input_values"],
        attention_mask=batch["attention_mask"]
    ).last_hidden_state

    initial_full_loss = float(
        original_forward.loss.item()
    )

# IMPORTANT: Clone OUTSIDE inference_mode.
# This creates a normal tensor usable during training.
frozen_features = (
    frozen_features.detach().clone().float()
)

# IMPORTANT: Copy the head OUTSIDE inference_mode.
initial_head = copy.deepcopy(
    baseline_model.lm_head
).to(DEVICE)

# Verify PyTorch can calculate gradients.
assert not frozen_features.is_inference()

for parameter in initial_head.parameters():
    assert parameter.requires_grad
    assert not parameter.is_inference()

print("AUTOGRAD TENSOR FIX: PASSED")

assert frozen_features.ndim == 3
assert frozen_features.shape[0] == 1

# Keep only actual target tokens, not padding.
targets = batch["labels"][
    batch["labels"] >= 0
].reshape(1, -1).contiguous()

assert targets.numel() > 0
assert not (targets == BLANK_ID).any().item()

input_lengths = torch.tensor(
    [frozen_features.shape[1]],
    dtype=torch.long
)

target_lengths = torch.tensor(
    [targets.shape[1]],
    dtype=torch.long
)

required_frames = (
    targets.shape[1]
    + (targets[0, 1:] == targets[0, :-1]).sum().item()
)

assert frozen_features.shape[1] >= required_frames

print("\nCACHED FEATURES")
print("Feature shape:", tuple(frozen_features.shape))
print("Target tokens:", targets.shape[1])
print("Required CTC frames:", required_frames)
print("Available frames:", frozen_features.shape[1])
print("Original model CTC loss:", round(initial_full_loss, 4))

# We no longer need the full pretrained encoder.
del baseline_model, original_forward
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 4. TRAIN ONLY THE ORIGINAL CTC HEAD
# ==========================================

head = initial_head.to(DEVICE)
head.train()

optimizer_probe = torch.optim.AdamW(
    head.parameters(),
    lr=HEAD_ONLY_LR,
    weight_decay=0.0
)

def head_only_loss():
    logits = head(frozen_features)

    # Use FP32 CTC to avoid mixed-precision
    # instability in this controlled experiment.
    log_probs = F.log_softmax(
        logits.float(),
        dim=-1
    ).transpose(0, 1)

    return F.ctc_loss(
        log_probs,
        targets,
        input_lengths,
        target_lengths,
        blank=BLANK_ID,
        reduction="mean",
        zero_infinity=False
    )


def evaluate_head():

    head.eval()

    with torch.no_grad():

        logits = head(frozen_features)

        probabilities = logits.float().softmax(
            dim=-1
        )

        predicted_ids = logits.argmax(dim=-1)

        predicted = processor.batch_decode(
            predicted_ids.cpu()
        )[0]

        predicted = normalize_amharic_ctc(
            predicted
        )

        blank_probability = (
            probabilities[..., BLANK_ID]
            .mean().item()
        )

        blank_argmax = (
            (predicted_ids == BLANK_ID)
            .float().mean().item()
        )

        loss = float(head_only_loss().item())

    return {
        "ctc_loss": loss,
        "cer": float(cer(REFERENCE, predicted)),
        "blank_probability": blank_probability,
        "blank_argmax": blank_argmax,
        "prediction": predicted
    }

# Sanity check: the cached-feature loss
# should match the complete starting model.
initial = evaluate_head()

print("\nLOSS CONSISTENCY CHECK")
print("Original model:", round(initial_full_loss, 5))
print("Cached features:", round(initial["ctc_loss"], 5))

assert abs(
    initial_full_loss - initial["ctc_loss"]
) < 0.05, "Cached-feature loss mismatch"

print("Consistency check: PASSED")

# ==========================================
# 5. RUN THE MEMORIZATION TEST
# ==========================================

history = []

for step in range(HEAD_ONLY_STEPS + 1):

    if step > 0:

        head.train()

        optimizer_probe.zero_grad(set_to_none=True)

        loss = head_only_loss()

        assert torch.isfinite(loss).item(), (
            f"Non-finite loss at step {step}"
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            head.parameters(),
            max_norm=5.0,
            error_if_nonfinite=True
        )

        optimizer_probe.step()

    if step in REPORT_STEPS:

        result = evaluate_head()

        history.append({
            "step": step,
            **result
        })

        print(f"\nSTEP {step}/{HEAD_ONLY_STEPS}")
        print(
            "CTC loss:",
            round(result["ctc_loss"], 4)
        )
        print(
            "CER:",
            f"{result['cer'] * 100:.2f}%"
        )
        print(
            "Blank probability:",
            f"{result['blank_probability'] * 100:.2f}%"
        )
        print(
            "Blank argmax:",
            f"{result['blank_argmax'] * 100:.2f}%"
        )
        print(
            "Prediction:",
            result["prediction"][:200] or "[EMPTY]"
        )

        if result["cer"] <= 0.05:
            print("\nMEMORIZATION TARGET REACHED")
            break

# ==========================================
# 6. SAVE DIAGNOSTIC RESULTS
# ==========================================

HEAD_ONLY_RESULTS = (
    DATA_DIR / "head_only_step0_memorization.json"
)

report = {
    "starting_checkpoint": str(BASELINE_DIR),
    "recording_id": sample_record["id"],
    "reference": REFERENCE,
    "trainable_component": "lm_head_only",
    "cached_encoder_features": True,
    "learning_rate": HEAD_ONLY_LR,
    "maximum_steps": HEAD_ONLY_STEPS,
    "completed_steps": step,
    "history": history
}

with HEAD_ONLY_RESULTS.open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        report,
        file,
        indent=2,
        ensure_ascii=False
    )

# ==========================================
# 7. FINAL COMPARISON
# ==========================================

final = history[-1]

print("\n" + "=" * 65)
print("HEAD-ONLY MEMORIZATION RESULTS")
print("=" * 65)

print("Starting loss:", round(initial["ctc_loss"], 4))
print("Final loss:", round(final["ctc_loss"], 4))

print(
    "Starting CER:",
    f"{initial['cer'] * 100:.2f}%"
)
print(
    "Final CER:",
    f"{final['cer'] * 100:.2f}%"
)

print("\nREFERENCE:")
print(REFERENCE)

print("\nFINAL PREDICTION:")
print(final["prediction"] or "[EMPTY]")

print("\nResults saved:", HEAD_ONLY_RESULTS)

# ==========================================
# 8. CLEAN UP
# ==========================================

del head, initial_head, optimizer_probe
del frozen_features, batch, targets

gc.collect()
torch.cuda.empty_cache()

print("\nMAIN EXPERIMENT UNCHANGED")
print("Head-only diagnostic completed independently.")
print("Existing training checkpoints were not modified.")


## Original source cell 36

Position 37 in the original notebook; original code comments may use different numbering.


In [ ]:

# =====================================================
# CELL 36 - EXTENDED HEAD-ONLY MEMORIZATION
# Goal: Can frozen XLS-R features reach 0% CER?
# Saves resumable checkpoints every 250 updates.
# Does not modify the main step-500 model.
# =====================================================

import copy
import json
import gc
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

MAX_STEPS = 4000
REPORT_EVERY = 250
LEARNING_RATE = 1e-3

TARGET_ID = "6Q0fEziAqyQVIz4J8aws"
BLANK_ID = tokenizer.pad_token_id
REFERENCE = sample_record["transcription"]

assert sample_record["id"] == TARGET_ID
assert sample_example is not None
assert torch.is_grad_enabled()

SAVE_DIR = DATA_DIR / "head_only_extended"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = SAVE_DIR / "latest.pt"
HISTORY_PATH = SAVE_DIR / "history.json"

device = torch.device(DEVICE)

# -----------------------------------------------------
# 1. Cache features from the ORIGINAL pretrained model
# -----------------------------------------------------

print("Loading original XLS-R checkpoint...")

original = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(device)

original.eval()

probe_batch = ctc_collator([sample_example])
probe_batch = {
    key: value.to(device)
    for key, value in probe_batch.items()
}

with torch.no_grad():
    original_loss = float(
        original(**probe_batch).loss.item()
    )

    features = original.wav2vec2(
        input_values=probe_batch["input_values"],
        attention_mask=probe_batch["attention_mask"]
    ).last_hidden_state

features = features.detach().clone().float()

# Copy trainable head outside any inference context.
probe_head = copy.deepcopy(
    original.lm_head
).to(device).float()

assert not features.is_inference()
assert probe_head.weight.requires_grad

del original
gc.collect()
torch.cuda.empty_cache()

# -----------------------------------------------------
# 2. Prepare CTC labels
# -----------------------------------------------------

probe_targets = probe_batch["labels"][
    probe_batch["labels"] >= 0
].reshape(1, -1).contiguous()

input_lengths = torch.tensor(
    [features.shape[1]], dtype=torch.long
)
target_lengths = torch.tensor(
    [probe_targets.shape[1]], dtype=torch.long
)

assert probe_targets.shape[1] > 0
assert features.shape[1] >= probe_targets.shape[1]

probe_optimizer = torch.optim.AdamW(
    probe_head.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0
)

def probe_loss():
    logits = probe_head(features)
    log_probs = F.log_softmax(
        logits.float(), dim=-1
    ).transpose(0, 1)

    return F.ctc_loss(
        log_probs,
        probe_targets,
        input_lengths,
        target_lengths,
        blank=BLANK_ID,
        reduction="mean",
        zero_infinity=False
    )

def probe_evaluate():
    probe_head.eval()

    with torch.no_grad():
        logits = probe_head(features)
        predicted_ids = logits.argmax(-1)

        prediction = processor.batch_decode(
            predicted_ids.cpu()
        )[0]

        prediction = normalize_amharic_ctc(
            prediction
        )

        loss = float(probe_loss().item())

        blank_fraction = (
            (predicted_ids == BLANK_ID)
            .float().mean().item()
        )

    return {
        "loss": loss,
        "cer": float(cer(REFERENCE, prediction)),
        "wer": float(wer(REFERENCE, prediction)),
        "blank_argmax": blank_fraction,
        "prediction": prediction
    }

# -----------------------------------------------------
# 3. Resume previous extended experiment if available
# -----------------------------------------------------

history = []
start_step = 0

if CHECKPOINT_PATH.exists():
    saved = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=True
    )

    assert saved["recording_id"] == TARGET_ID

    probe_head.load_state_dict(saved["head"])
    probe_optimizer.load_state_dict(saved["optimizer"])

    start_step = int(saved["step"])

    if HISTORY_PATH.exists():
        history = json.loads(
            HISTORY_PATH.read_text(encoding="utf-8")
        )

    print(f"Resuming from step {start_step}")

else:
    # Verify cached features give the same initial loss.
    initial_loss = float(probe_loss().item())

    assert abs(initial_loss - original_loss) < 0.05, (
        "Initial loss differs from original model."
    )

    print("Initial loss consistency: PASSED")

def save_progress(step, result):
    checkpoint = {
        "recording_id": TARGET_ID,
        "step": step,
        "head": probe_head.state_dict(),
        "optimizer": probe_optimizer.state_dict()
    }

    temporary = SAVE_DIR / "latest.pt.partial"
    torch.save(checkpoint, temporary)
    temporary.replace(CHECKPOINT_PATH)

    temporary_history = SAVE_DIR / "history.json.partial"

    temporary_history.write_text(
        json.dumps(
            history, ensure_ascii=False, indent=2
        ),
        encoding="utf-8"
    )
    temporary_history.replace(HISTORY_PATH)

# -----------------------------------------------------
# 4. Train, evaluate and checkpoint
# -----------------------------------------------------

print("\nEXTENDED MEMORIZATION EXPERIMENT")
print("-" * 55)
print("Recording:", TARGET_ID)
print("Training from step:", start_step)
print("Maximum step:", MAX_STEPS)
print("Frozen feature shape:", tuple(features.shape))

current = probe_evaluate()

print(f"\nSTEP {start_step}")
print("Loss:", round(current["loss"], 4))
print("CER:", f"{current['cer'] * 100:.2f}%")
print("Prediction:", current["prediction"] or "[EMPTY]")

if not history:
    history.append({"step": start_step, **current})

completed_step = start_step

for step in range(start_step + 1, MAX_STEPS + 1):

    probe_head.train()
    probe_optimizer.zero_grad(set_to_none=True)

    loss = probe_loss()

    assert torch.isfinite(loss).item()
    assert loss.requires_grad

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        probe_head.parameters(),
        max_norm=5.0,
        error_if_nonfinite=True
    )

    probe_optimizer.step()
    completed_step = step

    if step % REPORT_EVERY == 0 or step == MAX_STEPS:

        result = probe_evaluate()

        history.append({"step": step, **result})
        save_progress(step, result)

        print(f"\nSTEP {step}/{MAX_STEPS}")
        print("CTC loss:", round(result["loss"], 4))
        print("CER:", f"{result['cer'] * 100:.2f}%")
        print("WER:", f"{result['wer'] * 100:.2f}%")
        print(
            "Blank argmax:",
            f"{result['blank_argmax'] * 100:.2f}%"
        )
        print(
            "Prediction:",
            result["prediction"] or "[EMPTY]"
        )

        if result["cer"] == 0.0:
            print("\nEXACT TRANSCRIPTION MEMORIZED!")
            break

# -----------------------------------------------------
# 5. Final result
# -----------------------------------------------------

final = probe_evaluate()

print("\n" + "=" * 55)
print("FINAL RESULT")
print("=" * 55)
print("Completed steps:", completed_step)
print("Final CER:", f"{final['cer'] * 100:.2f}%")
print("Final WER:", f"{final['wer'] * 100:.2f}%")

print("\nREFERENCE:")
print(REFERENCE)

print("\nPREDICTION:")
print(final["prediction"] or "[EMPTY]")

print("\nCheckpoint:", CHECKPOINT_PATH)
print("History:", HISTORY_PATH)

print("\nOriginal baseline and step-500 model unchanged.")


## Original source cell 37

Position 38 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 37 - CONTROLLED ENCODER COMPARISON
#
# Question:
# Did fine-tuning to step 500 make the encoder's
# features harder to learn from?
#
# CONTROL:
# Step-0 encoder + step-0 head -> 0% CER at 2250 steps
#
# TEST:
# Step-500 encoder + SAME step-0 head initialization
#
# Both encoders remain frozen.
# No existing checkpoint will be modified.
# ============================================================

import copy
import json
import gc
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

CONTROL_MAX_STEPS = 2500
CONTROL_LR = 1e-3
CONTROL_REPORT_EVERY = 250

CONTROL_ID = "6Q0fEziAqyQVIz4J8aws"
CONTROL_BLANK = tokenizer.pad_token_id

STEP500_PATH = CHECKPOINT_DIR / "step_000500"

CONTROL_DIR = DATA_DIR / "encoder_step500_head_control"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_CKPT = CONTROL_DIR / "latest.pt"
CONTROL_HISTORY = CONTROL_DIR / "history.json"

ORIGINAL_HISTORY = (
    DATA_DIR / "head_only_extended" / "history.json"
)

assert Path(BASELINE_DIR).is_dir()
assert STEP500_PATH.is_dir()
assert ORIGINAL_HISTORY.is_file()
assert sample_record["id"] == CONTROL_ID
assert sample_example is not None
assert torch.is_grad_enabled()

control_device = torch.device(DEVICE)
control_reference = sample_record["transcription"]

# ------------------------------------------------------------
# 1. Prepare exactly the same recording and labels
# ------------------------------------------------------------

control_batch = ctc_collator([sample_example])

control_batch = {
    k: v.to(control_device)
    for k, v in control_batch.items()
}

control_targets = control_batch["labels"][
    control_batch["labels"] >= 0
].reshape(1, -1).contiguous()

control_target_lengths = torch.tensor(
    [control_targets.shape[1]], dtype=torch.long
)

# ------------------------------------------------------------
# 2. Load original encoder and save its original head
# ------------------------------------------------------------

print("Loading STEP-0 model...")

original_model = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(control_device)

original_model.eval()

with torch.no_grad():
    original_features = original_model.wav2vec2(
        input_values=control_batch["input_values"],
        attention_mask=control_batch["attention_mask"]
    ).last_hidden_state.detach().clone().float()

# Copy outside inference mode. Keep the exact starting head.
control_head = copy.deepcopy(
    original_model.lm_head
).to(control_device).float()

del original_model
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Get features from the STEP-500 encoder
# ------------------------------------------------------------

print("Loading STEP-500 model...")

step500_model = Wav2Vec2ForCTC.from_pretrained(
    str(STEP500_PATH)
).to(control_device)

step500_model.eval()

with torch.no_grad():
    step500_features = step500_model.wav2vec2(
        input_values=control_batch["input_values"],
        attention_mask=control_batch["attention_mask"]
    ).last_hidden_state.detach().clone().float()

del step500_model
gc.collect()
torch.cuda.empty_cache()

assert original_features.shape == step500_features.shape
assert not step500_features.is_inference()
assert control_head.weight.requires_grad

control_input_lengths = torch.tensor(
    [step500_features.shape[1]], dtype=torch.long
)

assert step500_features.shape[1] >= (
    control_targets.shape[1]
    + (
        control_targets[0, 1:]
        == control_targets[0, :-1]
    ).sum().item()
)

# ------------------------------------------------------------
# 4. Verify original experimental conditions
# ------------------------------------------------------------

def control_ctc_loss(features):
    logits = control_head(features)

    return F.ctc_loss(
        F.log_softmax(
            logits.float(), dim=-1
        ).transpose(0, 1),
        control_targets,
        control_input_lengths,
        control_target_lengths,
        blank=CONTROL_BLANK,
        reduction="mean",
        zero_infinity=False
    )

with torch.no_grad():
    original_start_loss = float(
        control_ctc_loss(original_features).item()
    )

assert abs(original_start_loss - 33.24486) < 0.05, (
    "Starting head or original features do not match "
    "the previous controlled experiment."
)

del original_features
torch.cuda.empty_cache()

print("\nCONTROL VERIFICATION")
print("-" * 50)
print("Original starting loss:", round(original_start_loss, 5))
print("Expected:", 33.24486)
print("Verification: PASSED")
print("Step-500 feature shape:", tuple(step500_features.shape))

# ------------------------------------------------------------
# 5. Create independent head optimizer
# ------------------------------------------------------------

control_optimizer = torch.optim.AdamW(
    control_head.parameters(),
    lr=CONTROL_LR,
    weight_decay=0.0
)

def evaluate_control():

    control_head.eval()

    with torch.no_grad():
        logits = control_head(step500_features)
        predictions = logits.argmax(dim=-1)

        text = processor.batch_decode(
            predictions.cpu()
        )[0]

        text = normalize_amharic_ctc(text)

        current_loss = float(
            control_ctc_loss(step500_features).item()
        )

        blank_fraction = float(
            (predictions == CONTROL_BLANK)
            .float().mean().item()
        )

    return {
        "loss": current_loss,
        "cer": float(cer(control_reference, text)),
        "wer": float(wer(control_reference, text)),
        "blank_argmax": blank_fraction,
        "prediction": text
    }

# ------------------------------------------------------------
# 6. Load original encoder learning curve
# ------------------------------------------------------------

with ORIGINAL_HISTORY.open(encoding="utf-8") as f:
    original_history = json.load(f)

original_by_step = {
    int(row["step"]): row
    for row in original_history
}

assert original_by_step[2250]["cer"] == 0.0

# ------------------------------------------------------------
# 7. Resume this diagnostic if it was interrupted
# ------------------------------------------------------------

control_history = []
control_start = 0

if CONTROL_CKPT.exists():

    state = torch.load(
        CONTROL_CKPT,
        map_location=control_device,
        weights_only=True
    )

    assert state["recording_id"] == CONTROL_ID
    assert state["experiment"] == "step500_encoder_step0_head"

    control_head.load_state_dict(state["head"])
    control_optimizer.load_state_dict(state["optimizer"])

    control_start = int(state["step"])

    if CONTROL_HISTORY.exists():
        control_history = json.loads(
            CONTROL_HISTORY.read_text(encoding="utf-8")
        )

    print("\nResuming diagnostic from step", control_start)

def save_control(step):

    state = {
        "experiment": "step500_encoder_step0_head",
        "recording_id": CONTROL_ID,
        "step": step,
        "head": control_head.state_dict(),
        "optimizer": control_optimizer.state_dict()
    }

    temp = CONTROL_DIR / "latest.pt.partial"
    torch.save(state, temp)
    temp.replace(CONTROL_CKPT)

    history_temp = CONTROL_DIR / "history.json.partial"
    history_temp.write_text(
        json.dumps(
            control_history, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )
    history_temp.replace(CONTROL_HISTORY)

# ------------------------------------------------------------
# 8. Run controlled comparison
# ------------------------------------------------------------

print("\nENCODER COMPARISON")
print("=" * 60)
print("Same recording:", CONTROL_ID)
print("Same head initialization: YES")
print("Same LR:", CONTROL_LR)
print("Only difference: Encoder at step 0 vs step 500")
print("Training starts at:", control_start)

start_result = evaluate_control()

print("\nSTEP", control_start)
print("CTC loss:", round(start_result["loss"], 4))
print("CER:", f"{start_result['cer'] * 100:.2f}%")
print("Prediction:", start_result["prediction"] or "[EMPTY]")

if not control_history:
    control_history.append({
        "step": control_start,
        **start_result
    })

completed_step = control_start

if start_result["cer"] == 0.0:
    print("Memorization already achieved.")
else:
    for step in range(control_start + 1, CONTROL_MAX_STEPS + 1):

        control_head.train()
        control_optimizer.zero_grad(set_to_none=True)

        loss = control_ctc_loss(step500_features)

        assert torch.isfinite(loss).item()
        assert loss.requires_grad

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            control_head.parameters(),
            max_norm=5.0,
            error_if_nonfinite=True
        )

        control_optimizer.step()
        completed_step = step

        if step % CONTROL_REPORT_EVERY == 0:

            result = evaluate_control()

            control_history.append({
                "step": step,
                **result
            })

            save_control(step)

            print(f"\nSTEP {step}/{CONTROL_MAX_STEPS}")
            print(
                "Step-500 encoder CER:",
                f"{result['cer'] * 100:.2f}%"
            )

            original_result = original_by_step.get(step)

            if original_result is not None:
                print(
                    "Step-0 encoder CER:",
                    f"{original_result['cer'] * 100:.2f}%"
                )

            print("CTC loss:", round(result["loss"], 4))
            print(
                "Blank argmax:",
                f"{result['blank_argmax'] * 100:.2f}%"
            )
            print(
                "Prediction:",
                result["prediction"] or "[EMPTY]"
            )

            if result["cer"] == 0.0:
                print("\nEXACT TRANSCRIPTION MEMORIZED")
                break

# ------------------------------------------------------------
# 9. Final diagnostic
# ------------------------------------------------------------

final_result = evaluate_control()

print("\n" + "=" * 60)
print("CONTROLLED COMPARISON COMPLETE")
print("=" * 60)

print("Original encoder:")
print("  Reached 0% CER at step 2250")

print("\nStep-500 encoder:")
print("  Completed steps:", completed_step)
print(
    "  Final CER:",
    f"{final_result['cer'] * 100:.2f}%"
)
print(
    "  Final WER:",
    f"{final_result['wer'] * 100:.2f}%"
)
print("  Final loss:", round(final_result["loss"], 4))

print("\nREFERENCE:")
print(control_reference)

print("\nPREDICTION:")
print(final_result["prediction"] or "[EMPTY]")

print("\nSaved:", CONTROL_DIR)
print("Existing training checkpoints unchanged.")
